# Pilot 2025 CO2-solubility/O2 integrated calibration, estimability and DOE

This notebook documents the deterministic workflow used for the pilot-scale natural-must dataset. It starts from the reduced extended fermentation model already used in the previous notebooks, then replaces the purely empirical CO2 gas-flow lag with a dissolved-CO2 buffer model and a parsimonious macroscopic oxygen transition model.

Selected inherited secondary structure: `secondary_full_chem_o2fixed`.

Selected inherited aroma structure: `ea_ethanol_nlimited`.

Selected CO2 structure after the new benchmark: `solubility_o2_slow_transition`.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display
RESULTS = Path('results/co2_solubility_integrated_doe')
PLOTS = RESULTS / 'plots'
print(RESULTS.resolve())


## 1. Full model used as baseline

The core fermentation model tracks viable biomass \(X\), dead biomass \(X_d\), assimilable nitrogen \(N\), glucose \(G\), fructose \(F\), ethanol \(E\), and glycerol \(Gly\):

$$\frac{dX}{dt}=(\mu-k_d)X+u_X$$

$$\frac{dX_d}{dt}=k_dX$$

$$\frac{dN}{dt}=-q_N f_N(T,N)X+u_N$$

$$\frac{dG}{dt}=-\left(q_{XG}f_N+q_{EG}f_G+m\frac{G}{G+F}\right)X+u_G$$

$$\frac{dF}{dt}=-\left(q_{XF}f_N+q_{EF}f_F+m\frac{F}{G+F}\right)X+u_F$$

$$\frac{dE}{dt}=(\beta_G f_G+\beta_F f_F)X+u_E$$

$$\frac{dGly}{dt}=(\gamma_G f_G+\gamma_F f_F)X$$

The secondary layer keeps pyruvate, acetaldehyde, acetate, and oxygen as the reduced chemical-proxy states. In this pilot notebook, the CO2 block additionally uses a macro-O2 state to delay the effective anaerobic ethanol/CO2 source when the must is initially air-saturated. The aroma layer predicts retained liquid concentration and accumulated condenser-equivalent loss for ethyl acetate, isoamyl acetate, and ethyl octanoate.


### Data read and curation check

In [ ]:
pd.read_csv(RESULTS / 'co2_curation_decisions.csv')

In [ ]:
for path in sorted((PLOTS / 'data').glob('data_read_check_*.png')):
    display(Image(filename=str(path)))
for path in sorted((PLOTS / 'data').glob('co2_curated_*.png')):
    display(Image(filename=str(path)))


**Plain-language interpretation.** The workbook is read batch-by-batch, `25150` and `25151` CO2 files are excluded, and `25171` is restarted at the annotated `Pre reinoculo` sample. This makes the post-reinoculation segment the effective fermentation start instead of treating the non-viable inoculum period as model lag.

## 2. Initial simulation before CO2 reformulation

In [ ]:
pd.read_csv(RESULTS / 'initial_fit_metrics.csv').sort_values(['group','relative_rmse']).head(40)

In [ ]:
for path in sorted((PLOTS / 'initial_fit').glob('initial_fit_*.png')):
    display(Image(filename=str(path)))


**Plain-language interpretation.** This block is the reference fit before adding the dissolved-CO2 and macro-O2 states. If the model predicts CO2 release far earlier than the sensor while ethanol/sugar curves remain acceptable, the issue is not only gas-liquid solubility: the model is also assuming anaerobic ethanol/CO2 production from the start.

## 3. CO2 solubility, macro-O2 transition model and benchmark

The dissolved CO2 state is:

$$\frac{dC_{CO2,L}}{dt}=r_{CO2,prod}-r_{CO2,gas}$$

The original gas-source proxy is tied to ethanol production:

$$r_{CO2,prod}=\frac{44.01}{2\cdot46.07}r_E$$

The O2-gated candidates replace that source with:

$$r_{CO2,prod}^{eff}=r_{CO2,ferm}^{base}\left[f_C+(1-f_C)\phi_{ana}(O_2)\right]+r_{CO2,resp}$$

where the Crabtree floor \(f_C\) prevents aerobic conditions from fully shutting down fermentation at high sugar, and

$$\phi_{ana}(O_2)=\frac{K_{ana}^{n}}{K_{ana}^{n}+O_2^{n}}.$$

The macro-O2 state is initialized near air saturation for fresh must:

$$O_2(0)=O_2^*(T,E,G,F),$$

with a low effective fraction for batch 25171 because its new \(t=0\) is post-reinoculation after the original non-viable inoculum period. The state evolves as:

$$\frac{dO_2}{dt}=k_{La,O2}(O_2^*-O_2)-q_{O2}X\frac{O_2}{K_{O2}+O_2}.$$

The respiration contribution is kept stoichiometric and small:

$$r_{CO2,resp}=\frac{44.01}{32.00}\frac{q_{O2}X\,O_2}{1000(K_{O2}+O_2)}.$$

Gas release from the liquid still follows:

$$r_{CO2,gas}=k_{rel}\max(C_{CO2,L}-C^*_{CO2},0).$$

The saturation concentration is represented as a process correlation:

$$C^*_{CO2}=s_{CO2}\,1.69\exp[-0.032(T-20)]\exp(0.0016E)\exp[-0.0012(G+F)].$$

This is not a purely arbitrary lag. It is a two-stage physical/effective model: fresh must can contain oxygen that suppresses the anaerobic ethanol/CO2 source, then generated CO2 fills the dissolved pool before gas flow appears once the pool approaches supersaturation. The sensor reports \(L/min\), so each batch is allowed a linear scale factor from model specific release \((g/L/h)\) to measured gas flow.


In [ ]:
pd.read_csv(RESULTS / 'co2_model_selection_summary.csv')

In [ ]:
pd.read_csv(RESULTS / 'co2_metrics_selected.csv')

In [ ]:
pd.read_csv(RESULTS / 'o2_macro_diagnostics_selected.csv').head(30)

In [ ]:
for path in sorted((PLOTS / 'co2_benchmark').glob('co2_benchmark_*.png')):
    display(Image(filename=str(path)))
for path in sorted((PLOTS / 'o2_macro').glob('o2_macro_*.png')):
    display(Image(filename=str(path)))


**Plain-language interpretation.** The selected model is `solubility_o2_slow_transition`. Its benchmark row has BIC `136.07`, data WSSE `124.65`, and selection score `136.07`. The O2-gated candidates are accepted only if they improve early CO2 timing without requiring active parameter bounds or a purely empirical lag.

## 4. Integrated post-CO2 calibration and validation

In [ ]:
pd.read_csv(RESULTS / 'fit_integrated_selected.csv')

In [ ]:
pd.read_csv(RESULTS / 'theta_selected_integrated.csv', index_col=0).head(100)

In [ ]:
pd.read_csv(RESULTS / 'final_fit_metrics.csv').sort_values(['group','relative_rmse']).head(60)

In [ ]:
for path in sorted((PLOTS / 'final_fit').glob('final_fit_*.png')):
    display(Image(filename=str(path)))


**Plain-language interpretation.** The integrated fit adjusts the selected aroma parameters while keeping the step-0 CO2 solubility parameters fixed. This prevents aroma residuals from moving the physical CO2 buffer to a non-physical boundary. Good retained-aroma fit but poor condensate fit indicates a partition/stripping issue; poor retained and total fit indicates a synthesis-kinetics issue.

## 5. CO2/O2 FIM, eigenvalues and estimability

The full coupled FIM, including all aroma states, is computationally expensive because each finite-difference perturbation must reintegrate liquid/condensate aroma partition. For this O2-structure iteration, the notebook reports a focused CO2/O2 Fisher Information Matrix for the selected gas-transfer parameters. This is the correct diagnostic for deciding whether the new O2-gated gas model improves the online CO2 direction.

The current-data Fisher Information Matrix is computed by finite differences in log-parameter coordinates:

$$J_{:,j}\approx \frac{r(\theta_j e^{\Delta})-r(\theta_j e^{-\Delta})}{2\Delta}$$

$$F=J^TJ.$$

Eigenvalues close to zero indicate practically weak directions; the weakest eigenvectors show which parameter combinations are confounded. The full aroma-coupled FIM should be rerun later with an aroma-partition cache if the goal is to redesign the whole aroma campaign.


In [ ]:
json.load(open(RESULTS / 'co2_o2_target_parameters.json'))

In [ ]:
pd.read_csv(RESULTS / 'co2_o2_eigen_spectrum_current.csv').head(20)

In [ ]:
pd.read_csv(RESULTS / 'co2_o2_weak_directions_current.csv')

In [ ]:
pd.read_csv(RESULTS / 'co2_o2_parameter_estimability_current.csv')

In [ ]:
display(Image(filename=str(PLOTS / 'fim' / 'co2_o2_current_relative_eigenvalues.png')))

**Plain-language interpretation.** This focused FIM answers a narrower question than the previous global FIM: can the online CO2 data distinguish the selected gas-release parameters after adding macro-O2 gating? If this block is well-conditioned but aroma fits remain weak, the remaining limitation is not the CO2/O2 gas timing alone.

## 6. CO2/O2 model-based DOE

Candidate natural-must experiments add an experiment FIM to the current-data FIM:

$$F_{total}=F_{current}+\sum_i F_i.$$

The ranking reports D-optimality through \(\log\det(F)\), E-optimality through the minimum eigenvalue, and a hybrid score:

$$\Phi_{hybrid}=\log\det(F)+2\log(\lambda_{min}/\lambda_{max})-0.05\log(\mathrm{trace}(F^{-1})).$$

This focused DOE uses the same additive multi-experiment logic used in the Dowling/Pyomo DoE examples: the current online CO2 data act as prior information and each candidate design contributes incremental information about the selected CO2/O2 gas-transfer parameters. It is not a replacement for a full aroma-coupled MBDoE.


In [ ]:
pd.read_csv(RESULTS / 'co2_o2_candidate_ranking.csv').head(15)

In [ ]:
pd.read_csv(RESULTS / 'co2_o2_selected_campaign_hybrid.csv')

In [ ]:
pd.read_csv(RESULTS / 'co2_o2_eigen_spectrum_current_plus_campaign.csv').head(20)

In [ ]:
for path in sorted((PLOTS / 'co2_o2_designs').glob('design_*.png')):
    display(Image(filename=str(path)))


**Plain-language interpretation.** The selected campaign begins with: not computed. Designs are chosen because they improve the weakest information directions after accounting for the current pilot data, not because their curves look intuitively different.